# Chapter 29: Stability of the synchronous state

Apply one common decaying inhibitory conductance to ten LIF neurons distributed
uniformly in time along their unperturbed cycle. The source experiment stops
each trajectory at its first spike. The shared Brian protocol keeps continuous
voltage dynamics and records first crossings; plots below truncate there.

Reference: [Python chapter 29](../python/chapter29.ipynb),
`simulate_lif_pulse_panels`. Coverage: no pulse, weak/slow (g=0.15,tau=9ms),
and strong/fast (g=2,tau=1ms) pulses with tau_m=10ms, I=0.12.
RTM timing maps, condition-number sweeps, and river geometry remain in Python.

Run from a source checkout after `uv sync --all-groups`, with this checkout's
`.venv` kernel. The new shared helpers are not assumed to be on PyPI.


In [ ]:
from pathlib import Path
import sys
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "mnd").is_dir())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import numpy as np
import matplotlib.pyplot as plt
from mnd.brian.phases import theta_prc, phase_pair, weak_difference, lif_inhibition


In [ ]:
conditions = [(0., 1., 'No inhibition'), (.15, 9., 'Weak / slow'), (2., 1., 'Strong / fast')]
results = [lif_inhibition(strength=g, tau_syn=tau) for g, tau, _ in conditions]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharey=True)
for ax, result, (_, _, title) in zip(axes, results, conditions):
    assert np.isfinite(result.v).all() and np.isfinite(result.first_spikes).all()
    for cell, first in enumerate(result.first_spikes):
        window = result.t <= first
        ax.plot(result.t[window], result.v[cell, window], lw=1)
    ax.axhline(1, color='k', ls=':', lw=.8)
    ax.set(xlabel='t [ms]', title=title, ylim=(0, 1.05))
axes[0].set_ylabel('normalized voltage')
plt.tight_layout()
plt.show()
print('First-spike timing spread [ms]:', [float(np.ptp(result.first_spikes)) for result in results])